## Laptop training checklist (before kicking off a real run)

**Notebook vs. script**
- Notebooks: fine for exploration and runs under ~10-15 minutes.
- Anything longer → move to a plain `.py` script run from the terminal.
  Reason: Jupyter kernels can silently die/hang, and don't survive
  a reboot or crash gracefully. Scripts + checkpointing do.

**Run it in tmux so it survives a closed terminal**
```bash
sudo apt install tmux      # one-time install
tmux new -s train
python3 train.py --epochs 100
# detach: Ctrl+b then d
# reattach later: tmux attach -t train
```

**Prevent the laptop from sleeping mid-run**
```
systemd-inhibit --what=sleep:idle python3 train.py
```
- Built into systemd already — nothing to install.
- Also disable lid-close suspend if training with the lid closed:
  edit `/etc/systemd/logind.conf`, set `HandleLidSwitch=ignore`,
  then `sudo systemctl restart systemd-logind`.
- Check GNOME Settings → Power for auto-suspend/screen-blank too.

**Power**
- Always train plugged in — on battery the GPU throttles hard.
- NVIDIA Settings → PowerMizer → "Prefer Maximum Performance"
  to stop it downclocking between bursts of activity.

**Thermals (Max-Q chips throttle under sustained load)**
- Elevate the laptop / use a cooling stand for long runs.
- Monitor live: `nvidia-smi -l 1` or `sudo apt install nvtop`
- Clocks dropping + temps pinned ~85-90°C = thermal throttling
  → fix is airflow, not code.

**Checkpoint every run — assume it can be interrupted**
```python
if os.path.exists("ckpt_latest.pt"):
    ckpt = torch.load("ckpt_latest.pt")
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
else:
    start_epoch = 0
```

**Log to a file since you won't be watching it live**
```
python3 train.py 2>&1 | tee train_log.txt
```

**Rule of thumb:** notebook for anything < 10 min, tmux + script +
checkpointing + plugged in + lid-suspend disabled for anything longer.